# CIFAR-10 — Rademacher One-Shot SGN
This notebook demonstrates VGG11 trained on CIFAR-10 with the **Theta** optimizer using the **Rademacher one-shot** heavy-tailed stochastic gradient noise (SGN) variant, proposed in [WOR22](https://arxiv.org/abs/2102.04297) and [WR25](https://arxiv.org/abs/2510.20905).

### TL; DR
> Instead of requiring multiple data loaders (SB, LB, SB_a, …), the Rademacher variant injects heavy-tailed noise **within a single mini-batch** by reweighting per-sample losses with balanced ±1 signs.

### Rademacher One-Shot SGN
$$
g_{heavy}
= 
\nabla_\theta
\left( 
\frac{1}{m}\sum_{i=1}^m 
(1+ Z s_i)\ell(\theta; z_i)
\right)
$$
- $Z$: Lomax-distributed heavy-tailed noise scalar
- $s_i \in \{+1, -1\}$: balanced Rademacher signs (half positive, half negative)
- $\varphi_b(\cdot)$: gradient norm clipping operator with clip threshold $b$; $\eta$: step size
- Only **one forward pass + one backward pass** per step (vs. three for multi-loader methods)

In [ ]:
import time
from itertools import cycle

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

from tqdm.auto import tqdm

from theta import Theta, NoiseStep

In [ ]:
torch.manual_seed(5959)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
transform = transforms.ToTensor()

train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

# Keep the demo quick. Remove the Subset wrappers for full CIFAR-10 runs.
#train_dataset = Subset(train_dataset, range(512))
#test_dataset = Subset(test_dataset, range(512))

# Only one training loader needed — Rademacher signs are applied within the batch.
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)

train_eval_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)  # sharpness eval
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print(f"train={len(train_dataset)}, test={len(test_dataset)}")

In [ ]:
def build_vgg11(num_classes=10):
    model = models.vgg11(weights=None)
    model.avgpool = nn.AdaptiveAvgPool2d((1, 1))
    model.classifier = nn.Linear(512, num_classes)
    return model

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
def batch_to_device(batch, device):
    x, y = batch
    return x.to(device), y.to(device)


def loss_on_batch(model, batch, criterion, device):
    x, y = batch_to_device(batch, device)
    logits = model(x)
    loss = criterion(logits, y)
    acc = logits.argmax(dim=1).eq(y).float().mean().item() * 100.0
    return loss, acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    for batch in loader:
        x, y = batch_to_device(batch, device)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * y.size(0)
        total_correct += logits.argmax(dim=1).eq(y).sum().item()
        total += y.size(0)
    model.train()
    return total_loss / total, 100.0 * total_correct / total


def expected_sharpness(model, loader, criterion, device, delta=0.01, num_samples=5):
    model.eval()
    base_loss, _ = evaluate(model, loader, criterion, device)
    params = [p for p in model.parameters() if p.requires_grad]
    total_diff = 0.0

    for _ in range(num_samples):
        noises = []
        with torch.no_grad():
            for p in params:
                noise = torch.randn_like(p).mul_(delta)
                p.add_(noise)
                noises.append(noise)

        perturbed_loss, _ = evaluate(model, loader, criterion, device)
        total_diff += abs(perturbed_loss - base_loss)

        with torch.no_grad():
            for p, noise in zip(params, noises):
                p.sub_(noise)

    model.train()
    return total_diff / num_samples

## Rademacher One-Shot SGN
The Rademacher variant replaces the multi-loader SGN formulation with balanced per-sample sign reweighting inside a **single** mini-batch:
$$
\text{loss} = \frac{1}{m}\sum_{i=1}^m (1 + Z\,s_i)\,\ell(\theta;\,z_i)
$$
where $s_i \in \{+1,-1\}$ with exactly $m/2$ of each sign (balanced Rademacher).

This achieves the same noise injection effect as the multi-loader methods but requires only **one forward + one backward pass** per step.

In [ ]:
MAX_STEPS = 30_000
NOISE_STOP_STEP = 25_000
LEARNING_RATE = 0.01
CLIP_THRESHOLD = 0.25
TAIL_PROBABILITY = 0.9

#### Rademacher one-shot SGN
$$
g_{heavy}
= 
\nabla_\theta
\left( 
\frac{1}{m}\sum_{i=1}^m 
(1+ Z s_i)\ell(\theta; z_i)
\right)
$$

In [ ]:
rademacher_model = build_vgg11().to(device)
rademacher_optimizer = Theta(
    rademacher_model.parameters(),
    base=optim.SGD,
    lr=LEARNING_RATE,
    alpha=1.4,
    scale=0.5,
    clip=CLIP_THRESHOLD / LEARNING_RATE,
    tail_prob=0.0,
)

data_iter = cycle(train_loader)
start = time.perf_counter()

pbar = tqdm(range(MAX_STEPS), desc="A-3: Rademacher one-shot SGN | p=0.0")
for step in pbar:
    x, y = batch_to_device(next(data_iter), device)
    batch_size = y.size(0)

    noise = rademacher_optimizer.sample_noise(use_noise=(step < NOISE_STOP_STEP))
    z = noise.value if noise.active else 0.0

    # Balanced Rademacher signs: exactly half +1 and half -1
    signs = torch.cat([
        torch.ones(batch_size // 2, device=device),
        -torch.ones(batch_size // 2, device=device),
    ])[torch.randperm(batch_size, device=device)]

    logits = rademacher_model(x)
    per_sample_loss = F.cross_entropy(logits, y, reduction="none")
    loss = ((1.0 + z * signs) * per_sample_loss).mean()
    base_loss = per_sample_loss.mean()
    acc = logits.argmax(dim=1).eq(y).float().mean().item() * 100.0

    rademacher_optimizer.zero_grad()
    loss.backward()
    rademacher_optimizer.step()

    pbar.set_postfix(
        loss=f"{base_loss.item():.4f}",
        acc=f"{acc:.1f}",
        noise=int(noise.active),
        z=f"{noise.value:.3f}",
        cdf=f"{noise.cdf:.3f}",
    )

elapsed = time.perf_counter() - start
train_sharpness = expected_sharpness(rademacher_model, train_eval_loader, criterion, device, delta=0.01, num_samples=5)
test_loss, test_acc = evaluate(rademacher_model, test_loader, criterion, device)
print(
    f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}%, "
    f"train_expected_sharpness={train_sharpness:.6f}, elapsed={elapsed:.1f}s"
)

#### Rademacher one-shot SGN with thresholding
$$
g_{heavy}
= 
\nabla_\theta
\left( 
\frac{1}{m}\sum_{i=1}^m 
(1+ Z\cdot\mathbf{I}\{ Z > C \} s_i)\ell(\theta; z_i)
\right)
$$

In [ ]:
rademacher_threshold_model = build_vgg11().to(device)
rademacher_threshold_optimizer = Theta(
    rademacher_threshold_model.parameters(),
    base=optim.SGD,
    lr=LEARNING_RATE,
    alpha=1.4,
    scale=0.5,
    clip=CLIP_THRESHOLD / LEARNING_RATE,
    tail_prob=TAIL_PROBABILITY,
)

data_iter = cycle(train_loader)
start = time.perf_counter()

pbar = tqdm(range(MAX_STEPS), desc=f"B-5: Rademacher with thresholding | p={TAIL_PROBABILITY}")
for step in pbar:
    x, y = batch_to_device(next(data_iter), device)
    batch_size = y.size(0)

    noise = rademacher_threshold_optimizer.sample_noise(use_noise=(step < NOISE_STOP_STEP))
    z = noise.value if noise.active else 0.0

    # Balanced Rademacher signs: exactly half +1 and half -1
    signs = torch.cat([
        torch.ones(batch_size // 2, device=device),
        -torch.ones(batch_size // 2, device=device),
    ])[torch.randperm(batch_size, device=device)]

    logits = rademacher_threshold_model(x)
    per_sample_loss = F.cross_entropy(logits, y, reduction="none")
    loss = ((1.0 + z * signs) * per_sample_loss).mean()
    base_loss = per_sample_loss.mean()
    acc = logits.argmax(dim=1).eq(y).float().mean().item() * 100.0

    rademacher_threshold_optimizer.zero_grad()
    loss.backward()
    rademacher_threshold_optimizer.step()

    pbar.set_postfix(
        loss=f"{base_loss.item():.4f}",
        acc=f"{acc:.1f}",
        noise=int(noise.active),
        z=f"{noise.value:.3f}",
        cdf=f"{noise.cdf:.3f}",
    )

elapsed = time.perf_counter() - start
train_sharpness = expected_sharpness(rademacher_threshold_model, train_eval_loader, criterion, device, delta=0.01, num_samples=5)
test_loss, test_acc = evaluate(rademacher_threshold_model, test_loader, criterion, device)
print(
    f"test_loss={test_loss:.4f}, test_acc={test_acc:.2f}%, "
    f"train_expected_sharpness={train_sharpness:.6f}, elapsed={elapsed:.1f}s"
)